# 04 — Realistic Negative Candidates

Проверка качества на более реалистичном множестве кандидатов: к положительным взаимодействиям добавляются непрослушанные треки, после чего основные подходы сравниваются на одном candidate set.

Также здесь оценивается collaborative filtering, в том числе отдельно для новых user-item пар.

---

## Исходное описание этапа

# Negatives evaluation — отдельный запуск

Этот ноутбук запускает только эксперимент с реалистичным множеством кандидатов из раздела 15+ исходного ноутбука. 
Дорогие этапы: агенты, Transformer, стекинг и предыдущие эксперименты — не выполняются.

**Важно:** для запуска в Kaggle нужен доступ к `base_listens.parquet` (обычно через Add Input / Output предыдущего ноутбука).

In [ ]:
import os, glob, json, time, gc, warnings
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import lightgbm as lgb

warnings.filterwarnings("ignore", message="Overriding the parameters")

SEED, N_USERS = 42, 1500
DAY, SHIFT, TRAIN_DAYS = 86400, 26, 60
SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working", "."]

def find_file(name, required=True):
    for root in SEARCH_ROOTS:
        hits = glob.glob(os.path.join(root, "**", name), recursive=True)
        if hits:
            return hits[0]
    if required:
        raise FileNotFoundError(
            f"не найден файл {name}: подключите Output предыдущей версии через Add Input"
        )
    return None

os.makedirs("results", exist_ok=True)

BASE = find_file("base_listens.parquet")
df = pd.read_parquet(BASE)

print("df:", df.shape, "| исходные колонки:", list(df.columns))

df["pr"] = (np.minimum(df["played_ratio_pct"], 100) / 100).astype(np.float32)
df["skip"] = (df["pr"] < 0.1).astype(np.float32)
df["full"] = (df["pr"] >= 0.9).astype(np.float32)

t_val, t_test = df["timestamp"].quantile([0.8, 0.9])
chk = np.where(
    df["timestamp"] < t_val,
    "train",
    np.where(df["timestamp"] < t_test, "val", "test")
)
assert (chk == df["split"].to_numpy()).all(), "разбиение не совпадает с сохранённым"

# ВАЖНО: negatives_eval использует artist_id и album_id.
# В исходном ноутбуке они восстанавливаются из mapping-файлов.
def load_map(fname):
    p = find_file(fname, required=False)
    if p is None:
        from huggingface_hub import hf_hub_download
        p = hf_hub_download(
            repo_id="yandex/yambda",
            repo_type="dataset",
            filename=fname,
            local_dir="data"
        )
    m = pq.read_table(p).to_pandas()
    other = [c for c in m.columns if c != "item_id"]
    assert len(other) == 1
    return m.drop_duplicates("item_id").set_index("item_id")[other[0]]

for col, fname_ in [
    ("artist_id", "artist_item_mapping.parquet"),
    ("album_id", "album_item_mapping.parquet"),
]:
    s = df["item_id"].map(load_map(fname_)).astype("float64")
    df[col] = np.where(
        s.isna(),
        -1 - df["item_id"].astype(np.int64),
        s
    ).astype(np.int64)

print(
    "артистов/альбомов:",
    df["artist_id"].nunique(),
    df["album_id"].nunique()
)

# Те же train/val и user-level sel/rep, что в исходном ноутбуке.
tr = (
    (df["split"] == "train")
    & (df["timestamp"] >= t_val - TRAIN_DAYS * DAY)
)
va = df["split"] == "val"

rs = np.random.default_rng(123)
val_users = df.loc[va, "uid"].unique()
half = set(
    rs.choice(
        val_users,
        size=len(val_users) // 2,
        replace=False
    ).tolist()
)
is_sel = df["uid"].isin(half)

print(
    "train:", int(tr.sum()),
    "| val:", int(va.sum()),
    "| sel:", int((va & is_sel).sum()),
    "| rep:", int((va & ~is_sel).sum())
)


## 1. Загрузка исходных данных и подготовка минимального состояния


In [ ]:
BASE = find_file("base_listens.parquet")
df = pd.read_parquet(BASE)
print("df:", df.shape)
print("колонки:", list(df.columns))

# Эти поля нужны для совместимости с negatives_eval.py.
df["pr"] = (np.minimum(df["played_ratio_pct"], 100) / 100).astype(np.float32)
df["skip"] = (df["pr"] < 0.1).astype(np.float32)
df["full"] = (df["pr"] >= 0.9).astype(np.float32)

t_val = df["timestamp"].quantile(0.8)
t_test = df["timestamp"].quantile(0.9)
chk = np.where(df["timestamp"] < t_val, "train", np.where(df["timestamp"] < t_test, "val", "test"))
assert (chk == df["split"].to_numpy()).all(), "разбиение не совпадает с сохранённым"

tr = (df["split"] == "train") & (df["timestamp"] >= t_val - TRAIN_DAYS * DAY)
va = df["split"] == "val"

rs = np.random.default_rng(123)
val_users = df.loc[va, "uid"].unique()
half = set(rs.choice(val_users, size=len(val_users) // 2, replace=False).tolist())
is_sel = df["uid"].isin(half)

print("train:", int(tr.sum()), "| val:", int(va.sum()))
print("sel пользователей:", len(half), "| rep пользователей:", len(val_users) - len(half))

## 2. `negatives_eval.py`

Ниже сохранена реализация из исходного ноутбука. Она сама строит признаки, добавляет непрослушанные кандидаты и обучает LightGBM.

In [ ]:
# =====================================================================
# negatives_eval.py  --  приоритет 3: реалистичное множество кандидатов
#
# Проблема (раздел 8 отчёта): кандидаты = только прослушанные треки, поэтому NDCG
# измеряет ранжирование ВНУТРИ прослушанного. Здесь к каждой группе (пользователь, день)
# добавляются n_neg непрослушанных за этот день треков (половина равномерно из каталога,
# половина пропорционально популярности), и ручной набор, полный пул и RFE сравниваются
# на одинаковых кандидатах по NDCG@10 и Recall@10.
#
# Самодостаточный модуль (не требует глобалов ноутбука). Проверен на синтетике
# (наивный перебор для признаков, инварианты негативов, сквозной запуск).
#
# Использование в ноутбуке 3 (после того как есть df, tr, va, is_sel, t_val):
#   from negatives_eval import run_negatives_experiment
#   res = run_negatives_experiment(df, tr.to_numpy(), (va & is_sel).to_numpy(), (va & ~is_sel).to_numpy(), n_neg=20)
# =====================================================================
import time, gc
import numpy as np, pandas as pd, lightgbm as lgb

DAY, SHIFT = 86400, 26
FAMILIES = {"user": ["uid"], "ui": ["uid", "item_id"], "uart": ["uid", "artist_id"], "ualb": ["uid", "album_id"],
            "item": ["item_id"], "art": ["artist_id"], "alb": ["album_id"]}
TARGETS = {"user": ("label", "pr"), "ui": ("label", "pr"), "uart": ("label", "pr"), "ualb": ("label", "pr"),
           "item": ("label",), "art": ("label",), "alb": ("label",)}
WINDOWS = (1, 7, 30, 90, 400)
TAG = {"label": "rate", "pr": "mpr", "skip": "skiprate", "full": "fullrate"}
LGB = dict(objective="lambdarank", metric="ndcg", eval_at=[10], learning_rate=0.05, num_leaves=63,
           min_data_in_leaf=200, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, verbosity=-1, num_threads=4)


# ---------------------------------------------------------------------------
# 1. Признаки «запрос против событий»: для строк q (в т.ч. синтетических негативов)
#    агрегаты по РЕАЛЬНЫМ событиям ev строго до начала суток запроса.
# ---------------------------------------------------------------------------
def query_window_feats(ev, q, keys, windows, targets, name, snap=DAY, shift=SHIFT):
    n = len(ev)
    codes = pd.concat([ev[keys], q[keys]], ignore_index=True).groupby(keys, sort=False).ngroup().to_numpy(np.int64)
    ce, cq = codes[:n], codes[n:]
    te = ev["timestamp"].to_numpy(np.int64)
    comb_e = (ce << shift) | te
    o = np.argsort(comb_e, kind="stable"); comb_s = comb_e[o]
    tq = (q["timestamp"].to_numpy(np.int64) // snap) * snap
    hi = np.searchsorted(comb_s, (cq << shift) | tq, side="left")
    cums = {t: np.concatenate([[0.0], np.cumsum(ev[t].to_numpy(np.float64)[o])]) for t in targets}
    out = {}
    for w in windows:
        lo = np.searchsorted(comb_s, (cq << shift) | np.maximum(tq - w * snap, 0), side="left")
        cnt = (hi - lo).astype(np.float64)
        out[f"{name}_cnt_{w}d"] = cnt.astype(np.float32)
        for t in targets:
            out[f"{name}_{TAG[t]}_{w}d"] = np.where(cnt > 0, (cums[t][hi] - cums[t][lo]) / np.maximum(cnt, 1), np.nan).astype(np.float32)
    return out


def build_features(ev, q, families=FAMILIES, targets=TARGETS, windows=WINDOWS):
    out = {}
    for name, keys in families.items():
        out.update(query_window_feats(ev, q, keys, windows, targets[name], name))
    return pd.DataFrame(out, index=q.index)


# ---------------------------------------------------------------------------
# 2. Негативы
# ---------------------------------------------------------------------------
def sample_negatives(pos, ev, catalog_end, n_neg, seed=0, day=DAY):
    """pos: позитивные строки сплита (uid, timestamp, item_id) с колонкой g.
    Каталог и популярность берутся только по событиям до catalog_end (без будущего).
    Негатив не должен совпадать ни с одним треком, прослушанным этим пользователем в этот день."""
    r = np.random.default_rng(seed)
    past = ev[ev["timestamp"] < catalog_end]
    cat, cnt = np.unique(past["item_id"].to_numpy(), return_counts=True)
    p_pop = cnt / cnt.sum()
    groups = np.unique(pos["g"].to_numpy()); G = len(groups); m = 2 * n_neg
    half = m // 2
    draw = np.empty((G, m), dtype=np.int64)
    draw[:, :half] = cat[r.integers(0, len(cat), (G, half))]
    draw[:, half:] = cat[r.choice(len(cat), (G, m - half), p=p_pop)]
    gg = np.repeat(groups, m); it = draw.ravel()
    M = int(max(ev["item_id"].max(), it.max())) + 1
    key_pos = pos["g"].to_numpy(np.int64) * M + pos["item_id"].to_numpy(np.int64)
    key = gg.astype(np.int64) * M + it
    keep = ~np.isin(key, key_pos)
    tab = pd.DataFrame({"g": gg[keep], "item_id": it[keep], "key": key[keep]}).drop_duplicates("key")
    tab = tab[tab.groupby("g").cumcount() < n_neg]
    attr = ev.drop_duplicates("item_id").set_index("item_id")[["artist_id", "album_id"]]
    tab = tab.join(attr, on="item_id")
    tab = tab.dropna(subset=["artist_id"])
    neg = pd.DataFrame({"uid": tab["g"].to_numpy() // 1000, "timestamp": (tab["g"].to_numpy() % 1000) * day + day // 2,
                        "item_id": tab["item_id"].to_numpy(), "artist_id": tab["artist_id"].to_numpy().astype(np.int64),
                        "album_id": tab["album_id"].to_numpy().astype(np.int64), "label": 0.0, "g": tab["g"].to_numpy(), "is_neg": 1})
    return neg


def build_split(df, mask, n_neg, seed, min_pos_rows=10):
    cols = ["uid", "timestamp", "item_id", "artist_id", "album_id", "label"]
    pos = df.loc[mask, cols].copy()
    pos["g"] = pos["uid"].astype(np.int64) * 1000 + pos["timestamp"].astype(np.int64) // DAY
    pos["is_neg"] = 0
    catalog_end = int(pos["timestamp"].min())
    neg = sample_negatives(pos, df, catalog_end, n_neg, seed)
    rows = pd.concat([pos, neg], ignore_index=True).sort_values("g", kind="stable").reset_index(drop=True)
    npos_rows = pos.groupby("g").size()
    rows["elig"] = rows["g"].map(npos_rows).ge(min_pos_rows).to_numpy()          # группы, как в исходном протоколе
    F = build_features(df, rows)
    return rows, F


# ---------------------------------------------------------------------------
# 3. Метрики и бутстрап
# ---------------------------------------------------------------------------
def group_metrics(g, s, y, k=10):
    d = pd.DataFrame({"g": np.asarray(g), "s": np.asarray(s, np.float64), "y": np.asarray(y, np.float64)})
    a = d.sort_values(["g", "s"], ascending=[True, False], kind="stable")
    ra = a.groupby("g").cumcount().to_numpy() + 1
    dcg = pd.Series(a["y"].to_numpy() / np.log2(ra + 1) * (ra <= k), index=a.index).groupby(a["g"]).sum()
    hits = pd.Series(a["y"].to_numpy() * (ra <= k), index=a.index).groupby(a["g"]).sum()
    b = d.sort_values(["g", "y"], ascending=[True, False], kind="stable")
    rb = b.groupby("g").cumcount().to_numpy() + 1
    idcg = pd.Series(b["y"].to_numpy() / np.log2(rb + 1) * (rb <= k), index=b.index).groupby(b["g"]).sum()
    npos = d.groupby("g")["y"].sum()
    out = pd.DataFrame({"ndcg": dcg / idcg, "recall": hits / npos})
    return out[npos > 0]


def boot_ci(m_a, m_b, col, n_boot=1000, seed=0):
    """Кластерный bootstrap по пользователям для разности (b - a) по группам."""
    diff = (m_b[col] - m_a[col]).dropna()
    pu = diff.groupby(diff.index // 1000).agg(["sum", "count"])
    sums, cnts = pu["sum"].to_numpy(), pu["count"].to_numpy()
    r = np.random.default_rng(seed); est = []
    for _ in range(n_boot):
        i = r.integers(0, len(sums), len(sums)); est.append(sums[i].sum() / cnts[i].sum())
    return float(diff.mean()), (float(np.percentile(est, 2.5)), float(np.percentile(est, 97.5)))


# ---------------------------------------------------------------------------
# 4. Модели
# ---------------------------------------------------------------------------
def _ds(rows, F, cols, ref=None):
    sz = rows.groupby("g", sort=True).size().to_numpy()
    return lgb.Dataset(F[cols].to_numpy(np.float32), label=rows["label"].to_numpy(), group=sz,
                       params={"max_bin": 63}, reference=ref, free_raw_data=False)


def fit_rank(R_tr, F_tr, R_sel, F_sel, cols, seed=0, rounds=300):
    dtr = _ds(R_tr, F_tr, cols); dsel = _ds(R_sel, F_sel, cols, dtr)
    m = lgb.train({**LGB, "seed": seed}, dtr, rounds, valid_sets=[dsel],
                  callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(0)])
    return m


def predict(m, F, cols):
    return m.predict(F[cols].to_numpy(np.float32), num_iteration=m.best_iteration or None)


def rfe_select(R_tr, F_tr, R_sel, F_sel, cols, K, drop=0.15, seed=0):
    cols = list(cols)
    while len(cols) > K:
        m = fit_rank(R_tr, F_tr, R_sel, F_sel, cols, seed=seed, rounds=220)
        imp = pd.Series(m.feature_importance("gain"), index=cols).sort_values(ascending=False)
        cols = imp.index[:max(K, int(len(cols) * (1 - drop)))].tolist()
    return cols


MANUAL_WINDOWS = (1, 7, 30)
def manual_cols(all_cols):
    keep = []
    for fam in ("user", "ui", "item"):
        for w in MANUAL_WINDOWS:
            keep += [f"{fam}_cnt_{w}d", f"{fam}_rate_{w}d"]
    return [c for c in keep if c in all_cols]


def run_negatives_experiment(df, mask_tr, mask_sel, mask_rep, n_neg=20, seed=0, K_rfe=None, out_csv="results/negatives_eval.csv"):
    """df: таблица прослушиваний с колонками uid, timestamp, item_id, artist_id, album_id, label, pr (как в ноутбуке).
    mask_*: булевы массивы по строкам df (train-окно, sel, rep). Возвращает таблицу метрик."""
    t0 = time.time()
    df = df[["uid", "timestamp", "item_id", "artist_id", "album_id", "label", "pr"]].reset_index(drop=True)
    splits = {}
    for nm, mk in [("tr", mask_tr), ("sel", mask_sel), ("rep", mask_rep)]:
        splits[nm] = build_split(df, np.asarray(mk), n_neg, seed + {"tr": 1, "sel": 2, "rep": 3}[nm])
        R_, F_ = splits[nm]
        print(f"[{nm}] строк {len(R_)}, негативов {int(R_['is_neg'].sum())}, групп {R_['g'].nunique()}, признаков {F_.shape[1]} | {round(time.time()-t0)} с", flush=True)
    (R_tr, F_tr), (R_sel, F_sel), (R_rep, F_rep) = splits["tr"], splits["sel"], splits["rep"]
    all_cols = list(F_tr.columns); man = manual_cols(all_cols); K = K_rfe or len(man)
    models = {"manual": man, "pool": all_cols}
    models[f"rfe{K}"] = rfe_select(R_tr, F_tr, R_sel, F_sel, all_cols, K, seed=seed)
    el = R_rep["elig"].to_numpy()
    pop_score = F_rep["item_cnt_30d"].fillna(0).to_numpy()
    mets = {"random": group_metrics(R_rep.loc[el, "g"], np.random.default_rng(0).random(el.sum()), R_rep.loc[el, "label"]),
            "popularity": group_metrics(R_rep.loc[el, "g"], pop_score[el], R_rep.loc[el, "label"])}
    for nm, cols in models.items():
        m = fit_rank(R_tr, F_tr, R_sel, F_sel, cols, seed=seed)
        mets[nm] = group_metrics(R_rep.loc[el, "g"], predict(m, F_rep.loc[el], cols), R_rep.loc[el, "label"])
        print(f"[{nm}] {len(cols)} признаков | NDCG@10 {mets[nm]['ndcg'].mean():.4f} | Recall@10 {mets[nm]['recall'].mean():.4f}", flush=True)
    rows = [dict(модель=k, признаков=len(models.get(k, [])), ndcg10=v["ndcg"].mean(), recall10=v["recall"].mean(), групп=len(v)) for k, v in mets.items()]
    for a, b in [("manual", "pool"), ("manual", f"rfe{K}"), ("pool", f"rfe{K}")]:
        for col in ("ndcg", "recall"):
            d_, ci = boot_ci(mets[a], mets[b], col)
            rows.append(dict(модель=f"{b} − {a}", метрика=col, d=d_, lo=ci[0], hi=ci[1]))
            print(f"  {b} − {a} [{col}]: {d_:+.4f} [{ci[0]:+.4f}; {ci[1]:+.4f}]")
    res = pd.DataFrame(rows)
    res.to_csv(out_csv, index=False)
    return res


In [ ]:
# Проверяем, есть ли нужные колонки
print("artist_id:", "artist_id" in df.columns)
print("album_id:", "album_id" in df.columns)

if "artist_id" not in df.columns or "album_id" not in df.columns:
    import pyarrow.parquet as pq

    def load_item_map(fname):
        p = find_file(fname, required=False)

        if p is None:
            from huggingface_hub import hf_hub_download
            p = hf_hub_download(
                repo_id="yandex/yambda",
                repo_type="dataset",
                filename=fname,
                local_dir="data",
            )

        m = pq.read_table(p).to_pandas()
        other = [c for c in m.columns if c != "item_id"]

        if len(other) != 1:
            raise ValueError(
                f"Неожиданная структура {fname}: {list(m.columns)}"
            )

        return (
            m.drop_duplicates("item_id")
             .set_index("item_id")[other[0]]
        )

    for col, fname in [
        ("artist_id", "artist_item_mapping.parquet"),
        ("album_id", "album_item_mapping.parquet"),
    ]:
        mapping = load_item_map(fname)

        s = df["item_id"].map(mapping).astype("float64")

        df[col] = np.where(
            s.isna(),
            -1 - df["item_id"].astype(np.int64),
            s,
        ).astype(np.int64)

print("Колонки после восстановления:")
print(list(df.columns))

print(
    "artist_id:", df["artist_id"].nunique(),
    "| album_id:", df["album_id"].nunique()
)

In [ ]:
# Финальный запуск.
# Ничего из Transformer/агентов здесь не запускается.
res = run_negatives_experiment(
    df,
    tr.to_numpy(),
    (va & is_sel).to_numpy(),
    (va & ~is_sel).to_numpy(),
    n_neg=20,
)
display(res)
